In [1]:
import pandas as pd
import numpy as np
import json
import re
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score
import joblib
import configparser

In [2]:
# Читаем гиперпараметры из config.ini
config = configparser.ConfigParser()
config.read('../config.ini')

MAX_ITER = int(config['model']['max_iter'])
C = float(config['model']['C'])
MAX_FEATURES = int(config['model']['max_features'])
TEST_SIZE = float(config['model']['test_size'])
RANDOM_STATE = int(config['model']['random_state'])

print("Гиперпараметры загружены:")
print(f"  max_iter={MAX_ITER}, C={C}, max_features={MAX_FEATURES}")

Гиперпараметры загружены:
  max_iter=1000, C=1.0, max_features=10000


In [3]:
# Загружаем данные
records = []
with open('../data/reviews_Digital_Music_5.json', 'r', encoding='utf-8') as f:
    for line in f:
        records.append(json.loads(line))

df = pd.DataFrame(records)
print(f"Всего отзывов: {len(df)}")
print(f"Колонки: {df.columns.tolist()}")
print(df[['reviewText', 'overall']].head(3))

Всего отзывов: 64706
Колонки: ['reviewerID', 'asin', 'reviewerName', 'helpful', 'reviewText', 'overall', 'summary', 'unixReviewTime', 'reviewTime']
                                          reviewText  overall
0  It's hard to believe "Memory of Trees" came ou...      5.0
1  A clasically-styled and introverted album, Mem...      5.0
2  I never thought Enya would reach the sublime h...      5.0


In [5]:
# Убираем пустые отзывы
df = df.dropna(subset=['reviewText', 'overall'])

# Преобразуем рейтинг в sentiment:
# 1-2 → negative (0), 3 → neutral (1), 4-5 → positive (2)
def get_sentiment(rating):
    if rating <= 2:
        return 0
    elif rating == 3:
        return 1
    else:
        return 2

df['label'] = df['overall'].apply(get_sentiment)

# Простая очистка текста
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['clean_text'] = df['reviewText'].apply(clean_text)

print(f"Данных после очистки: {len(df)}")
print(f"\nРаспределение классов:")
print(df['label'].value_counts().sort_index().rename({0:'negative', 1:'neutral', 2:'positive'}))

Данных после очистки: 64706

Распределение классов:
label
negative     5801
neutral      6789
positive    52116
Name: count, dtype: int64


In [6]:
# Разбиваем на train/test
X_train, X_test, y_train, y_test = train_test_split(
    df['clean_text'], df['label'],
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE
)

# Превращаем текст в числа через TF-IDF
vectorizer = TfidfVectorizer(max_features=MAX_FEATURES)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

# Обучаем модель
model = LogisticRegression(C=C, max_iter=MAX_ITER, random_state=RANDOM_STATE)
model.fit(X_train_tfidf, y_train)

# Метрики
y_pred = model.predict(X_test_tfidf)
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print("\nDetailed report:")
print(classification_report(y_test, y_pred, target_names=['negative', 'neutral', 'positive']))

Accuracy: 0.8524

Detailed report:
              precision    recall  f1-score   support

    negative       0.76      0.47      0.58      1200
     neutral       0.53      0.21      0.30      1350
    positive       0.87      0.98      0.92     10392

    accuracy                           0.85     12942
   macro avg       0.72      0.55      0.60     12942
weighted avg       0.83      0.85      0.83     12942



In [ ]:
import os

# Создаём папку для модели
os.makedirs('../experiments', exist_ok=True)

# Сохраняем модель и векторайзер
joblib.dump(model, '../experiments/model.pkl')
joblib.dump(vectorizer, '../experiments/vectorizer.pkl')

print("Модель сохранена: experiments/model.pkl")
print("Векторайзер сохранён: experiments/vectorizer.pkl")

# Проверка — загружаем и тестируем
loaded_model = joblib.load('../experiments/model.pkl')
loaded_vectorizer = joblib.load('../experiments/vectorizer.pkl')

test_reviews = [
    "This album is absolutely amazing, I love it!",
    "Terrible quality, waste of money.",
    "It's okay, nothing special."
]

for review in test_reviews:
    cleaned = clean_text(review)
    vec = loaded_vectorizer.transform([cleaned])
    pred = loaded_model.predict(vec)[0]
    labels = {0: 'negative', 1: 'neutral', 2: 'positive'}
    print(f"\n'{review}'\n→ {labels[pred]}")

Модель сохранена: experiments/model.pkl
Векторайзер сохранён: experiments/vectorizer.pkl

'This album is absolutely amazing, I love it!'
→ positive

'Terrible quality, waste of money.'
→ negative

'It's okay, nothing special.'
→ negative
